# SFT
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/sft.ipynb)

Use supervised fine-tuning (SFT) on isolated IDRs, save the model, and generate new sequences.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [ ]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record, read_records
from idiom.utils.notebook_helpers import (
    check_context,
    example_file,
    isolated,
    save_run,
    split_records,
    train_and_save,
    training_config,
    write_fasta,
)

started = time.perf_counter()


## Settings


In [ ]:
MODEL_ID = "jxliu2/idiom-300M"
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 2 # Training and generation batch size
SEED = 0
MAX_STEPS = 20 # Training steps for this demonstration
N = 8 # Number of standalone sequences to generate
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
OUT_DIR = Path("sft_outputs") / time.strftime("%Y%m%d-%H%M%S")

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Prepare sequences
Use the prepared nucleolus IDRs from ProtGPS. Split them into 80% training and 20% validation data; exact duplicates stay together. This split is not homology-aware.


In [ ]:
input_path = example_file("protgps/idrs/nucleolus_idrs.fasta", Path("example_inputs"))
records = isolated(list(read_records(input_path)))
print(f"Loaded {len(records)} nucleolus IDRs.")


In [ ]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
check_context(records, base.model.cfg.max_seq_len)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
train_records, validation_records = split_records(records, seed=SEED)
train_path = write_fasta(train_records, OUT_DIR / "train.fasta")
validation_path = write_fasta(validation_records, OUT_DIR / "validation.fasta")
print("Train:", len(train_records), "Validation:", len(validation_records))

## 2. Fine-tune the model
Fine-tune in unprompted mode with a short demonstration run. Increase `MAX_STEPS` and use your full dataset for a longer run. Training saves a reloadable model and local logs in `OUT_DIR`.


In [ ]:
cfg = training_config("sft", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.data.train_fasta = str(train_path.resolve())
cfg.data.val_fasta = str(validation_path.resolve())
cfg.data.prompted_prob = 0.0
cfg.data.batch_size = min(BATCH_SIZE, len(train_records))
cfg.data.num_workers = 0
cfg.optim.lr = 1e-5
cfg.optim.warmup_steps = min(5, MAX_STEPS)
release = train_and_save(cfg)

### Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [ ]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "val/loss"] if name in metrics]
display(metrics[columns].tail(3))

## 3. Generate from the fine-tuned model


In [ ]:
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
table = pd.DataFrame({"sequence": adapted, "length": [len(s) for s in adapted]})
table.to_csv(OUT_DIR / "candidates.csv", index=False)
display(table)

## Results
`candidates.csv` and `adapted.fasta` contain generated IDRs. `model/` is a reloadable IDiom release. `training/` contains logs and a checkpoint; `validation_metrics.json` records validation loss.

Download results from Colab’s Files pane before the runtime ends.


In [ ]:
save_run(
    OUT_DIR,
    dict(
        model=MODEL_ID,
        device=DEVICE,
        input=str(input_path),
        seed=SEED,
        steps=MAX_STEPS,
        sampling=sampling,
    ),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)